<a href="https://colab.research.google.com/github/rajidush/RP-J26-DS-316/blob/main/component3_socratic_educator/notebooks/simulate_children_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simulated children 8-15 (Component 3, ticket 06)

A model plays each child and answers every line of the **real** Component 3 controller, through
`simulate_children.py`. The run gives two pools:

- **classifier_test**: 120 sessions (40 per age band), one per classifier-test scenario of the PP2 draw. The raters label these replies (ticket 09).
- **train_dev**: 72 sessions (24 per band), one per training or validation scenario. You label these yourself for the fine-tuned arm (ticket 14); they are never evidence.

About 1 in 5 children is played by Gemma through AI Studio; the rest by Gemini through Colab.

Order: **plan → a 3-session trial → Gemini → Gemma → finish**. Every step is resumable: if a quota runs out, finished
sessions are kept, and re-running the same cell continues from where it stopped. Everything is saved to Google Drive.

In [ ]:
!pip -q install "openai>=1.40"

## Get the code

Clones `main`, so run this notebook after the `c3/simulated-children` branch is merged. If the repository is private,
upload a zip instead: on your laptop run `git archive -o c3.zip HEAD component3_socratic_educator docs/interface-contracts`
in the repo (the controller reads the contracts in `docs/`), upload `c3.zip` here, and run the second cell.

In [ ]:
import os
REPO = "/content/RP-J26-DS-316"
if not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/rajidush/RP-J26-DS-316.git {REPO}
%cd {REPO}/component3_socratic_educator

In [ ]:
# Only if the clone failed: unzip an uploaded c3.zip instead.
# from google.colab import files
# files.upload()
# !mkdir -p /content/c3 && unzip -o -q c3.zip -d /content/c3
# %cd /content/c3/component3_socratic_educator

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/c3_simulated_children"
DRAW = "evidence/inquire/pp2_scenario_draw_2026-11-04/scenario_draw.json"

## 1. Plan (once)

Draws each session's pool, scenario, band, archetype and generator before any child exists. If the plan is already on Drive, this cell refuses and keeps it.

In [ ]:
!python simulate_children.py plan --draw {DRAW} --out "{OUT}"

## 2. Check Gemini access through Colab

This also loads the Colab model-proxy credentials for the script.

In [ ]:
import os
from google.colab import ai
MODEL = "google/gemini-2.5-pro"   # or another Gemini model your account lists
print(ai.generate_text("Reply with the single word OK.", model_name=MODEL))
assert os.environ.get("MODEL_PROXY_HOST") and os.environ.get("MODEL_PROXY_API_KEY")

## 3. Trial: three Gemini sessions, one per band

The trial uses train/dev children, never held-out ones, so the prompt is never tuned on test sessions. Read them before the full run: do the replies sound like children of the band?

In [ ]:
!python simulate_children.py run --plan "{OUT}/children.json" --generator gemini --model "{MODEL}" --backend colab --pool train_dev --per-band 1

In [ ]:
import json, os
path = f"{OUT}/sessions.jsonl"
for line in (open(path, encoding="utf-8") if os.path.exists(path) else []):
    s = json.loads(line)
    print(f"\n{s['session_id']}  {s['age_band']}  {s['archetype']}  {s['risk_category']}")
    for t in s["turns"]:
        if t["child_response"] is not None:
            print("  educator:", t["system_text"])
            print("  child:   ", t["child_response"])

## 4. All Gemini children

Re-run this cell until it says `0 still to simulate`. A quota error stops it cleanly; wait, then re-run.

In [ ]:
!python simulate_children.py run --plan "{OUT}/children.json" --generator gemini --model "{MODEL}" --backend colab

## 5. The Gemma slice through AI Studio

Needs your free AI Studio key in Colab's secrets as `GEMINI_API_KEY`. `--fold-system` sends the child's persona in the
first message, because AI Studio's Gemma models reject a system message. Re-run until `0 still to simulate`.

In [ ]:
import os
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
from openai import OpenAI
client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=os.environ["GEMINI_API_KEY"])
print([m.id for m in client.models.list() if "gemma" in m.id])
GEMMA_MODEL = "gemma-3-27b-it"   # pick one from the list above

In [ ]:
!python simulate_children.py run --plan "{OUT}/children.json" --generator gemma --model "{GEMMA_MODEL}" --backend aistudio --fold-system

## 6. Finish

Checks every planned session is there, that no held-out child or scenario is in the train/dev pool, that every Component 4 record passes the contract and holds no raw reply; then writes one file per pool and `manifest.json`.

In [ ]:
!python simulate_children.py finish --dir "{OUT}" --draw {DRAW}
print(open(f"{OUT}/manifest.json", encoding="utf-8").read()[:3000])

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("/content/c3_simulated_children", "zip", OUT)
files.download("/content/c3_simulated_children.zip")

## After the run (on your laptop)

1. Unzip into `component3_socratic_educator/evidence/evaluate/simulated_children_<date>/` (the date of the run).
2. Raters' reply-label sheets: `python inquire_finetune.py label-sheets --sessions evidence/evaluate/simulated_children_<date>/classifier_test_sessions.jsonl`
3. Your own train/dev sheet: `python inquire_finetune.py label-sheets --owner --sessions evidence/evaluate/simulated_children_<date>/train_dev_sessions.jsonl`